### Radfahrspaß – Lösung Schritt für Schritt

*35. Bundeswettbewerb Informatik (2016), 1. Runde, Aufgabe 4*

Beim Indoor-Fahrspaßwettbewerb fährt man mit einem raketengetriebenen Spaßrad einen Parcours ab. Ein Parcours ist eine gerade Strecke aus Abschnitten: **bergab** (`\`, das Rad wird um 1 m/s schneller), **bergauf** (`/`, um 1 m/s langsamer) oder **flach** (`_`). Auf einem flachen Abschnitt muss der Fahrer das Rad um genau 1 m/s beschleunigen (`+`) oder verlangsamen (`–`). Man startet mit 0 m/s und muss am Ziel wieder mit genau 0 m/s ankommen; stehen bleiben mitten im Parcours (Geschwindigkeit 0 und dann noch ein Anstieg) oder zurückrollen ist verboten, die Geschwindigkeit darf also nie negativ werden.

Die Fahrer dürfen den Parcours vorher einmal zu Fuß abgehen, dürfen sich aber nichts notieren und können sich nur wenig merken. Danach sagen sie, ob sie ihn schaffen.

Die drei Beispiele der Aufgabe: A) `\_////`, B) `\_\_\\` und C) `\_\__/`. A und B sind nicht befahrbar (in A bleibt das Rad mittendrin stehen, in B hat es am Ziel noch mindestens 2 m/s), C schafft man mit `+ – –` auf den flachen Abschnitten.

#### Aufgabe 4

1. Entwickle ein möglichst einfaches Verfahren, mit dem die Fahrer beim Probelauf entscheiden können, ob der Parcours regelkonform befahrbar ist. Simuliere es mit einem Computer und verwende möglichst wenig Speicherplatz. Der Parcours wird nur **einmal** durchschritten und kann sehr lang sein: das Programm darf die Eingabe nur einmal lesen und nicht speichern.
2. Erweitere das Programm so, dass es bei einem **zweiten Einlesen** eines befahrbaren Parcours für jeden flachen Abschnitt eine Anweisung `+` oder `–` ausgibt. Wer sie befolgt, steht am Ziel genau still.

**Zu den Beispieldaten:** Die drei Beispiele A, B, C (`parcoursA.txt` bis `parcoursC.txt`) und `parcours0.txt` liegen im Ordner `beispieldaten`. Die zehn eigentlichen Testparcours `parcours0` bis `parcours9` sind riesig (bis zu 500 Millionen Zeichen, `parcours.zip` hat 195 MB) und gibt es nur auf der BWINF-Materialseite (https://bwinf.de/bundeswettbewerb/35/1/). Ich habe sie mit genau den Funktionen unten laufen lassen; das Ergebnis steht in einer Tabelle am Ende. Im Ordner `eigene_testdaten` liegt ein kleiner Parcours (`vmax_reicht_nicht.txt`), den wir im Abschnitt „Für Neugierige“ brauchen.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Menge der möglichen Zustände mitführen:** Weil der Fahrer auf flachen Stücken frei wählt, gibt es nach einem Abschnitt meist *mehrere* mögliche Geschwindigkeiten. Statt eine Entscheidung zu raten, merkt man sich **alle** möglichen Geschwindigkeiten. Das ist die Grundidee vieler Verfahren, bei denen man nicht zurückschauen darf.
- **Eine Menge durch zwei Zahlen beschreiben:** Die möglichen Geschwindigkeiten liegen immer im Abstand 2 hintereinander (`1, 3, 5, …`). Dann genügen der kleinste und der größte Wert, `vmin` und `vmax`. Das spart Speicher: egal wie lang der Parcours ist, man merkt sich nur ein paar Zahlen.
- **Datenstrom (Streaming):** Daten, die zu groß für den Speicher sind, liest man in kleinen Portionen und vergisst sie wieder. Der Zustand (hier `vmin`, `vmax`, Zahl der flachen Stücke) ist alles, was man mitnimmt.
- **Klammerfolgen:** Die Aufgabe ist dasselbe wie die Frage, ob man aus einer Folge von `(`, `)` und `?` durch Ersetzen jedes `?` durch eine Klammer eine korrekte Klammerung machen kann (Beispiel C: `(?(??)`). Der Zähler „offene Klammern“ ist die Geschwindigkeit.

**Nützliche Funktionen und Bibliotheken**
- `open(...)` und `datei.read(n)`: liest höchstens `n` Zeichen; wiederholt man das, bekommt man die Datei Portion für Portion, ohne sie ganz zu laden. Leere Zeichenkette heißt: Ende der Datei.
- `io.StringIO(text)`: tut so, als wäre ein Text eine Datei. Damit können wir dieselbe Funktion auch auf kurze Texte anwenden.
- `itertools.product`: alle Kombinationen, hier alle Folgen von `+` und `-`, um die Funktion durch „Alles probieren“ zu prüfen.

## Die Idee in Worten

Wir laufen den Parcours Zeichen für Zeichen ab und merken uns, welche Geschwindigkeiten der Fahrer an dieser Stelle **haben könnte**.

- Am Start ist nur 0 möglich.
- `\` (bergab): jede mögliche Geschwindigkeit wird um 1 größer.
- `/` (bergauf): jede wird um 1 kleiner.
- `_` (flach): jede Geschwindigkeit `v` kann zu `v + 1` oder `v - 1` werden. Aus der Menge `{1, 3}` wird `{0, 2, 4}`: der kleinste Wert sinkt um 1, der größte steigt um 1, und alles dazwischen bleibt im Abstand 2 lückenlos besetzt.

Also: bei `\` erhöhen wir `vmin` und `vmax` um 1, bei `/` verringern wir beide um 1, bei `_` wird `vmin` um 1 kleiner und `vmax` um 1 größer.

**Negative Geschwindigkeiten gibt es nicht.** Zwei Fälle:
1. Wird `vmax` kleiner als 0, dann ist *jede* mögliche Geschwindigkeit negativ: Das Rad bleibt auf jeden Fall stehen. **Nicht befahrbar.**
2. Wird `vmin` kleiner als 0 (dann ist es genau −1), fällt nur der kleinste Wert weg. Der nächste mögliche Wert ist 1, also setzen wir `vmin` um 2 höher.

Am Ende ist der Parcours befahrbar, wenn **0** möglich ist. Weil `vmin` der kleinste mögliche Wert ist und alle möglichen Werte dieselbe Parität (gerade/ungerade) haben, ist das genau der Fall, wenn `vmin == 0`.

**Teil 2:** Wer immer nur beschleunigt (`+`), kommt am Ende mit `vmax` an. Damit es 0 wird, müssen genau `vmax / 2` der `+` durch `–` ersetzt werden (jeder Tausch senkt die Endgeschwindigkeit um 2). Tauscht man die **letzten** `vmax / 2` flachen Abschnitte, bleibt die Geschwindigkeit unterwegs sicher nicht negativ, denn vorne wird ja so schnell wie möglich beschleunigt. Es genügt also, die flachen Abschnitte zu zählen: die ersten `flach - vmax / 2` bekommen `+`, die restlichen `–`.

## Schritt 1: Ein Parcours prüfen (Teil 1)

`pruefe(datei)` liest die geöffnete Datei in Portionen zu einer Million Zeichen (die Zeichen laufen dann einzeln durch die `for`-Schleife) und gibt drei Werte zurück: ob der Parcours befahrbar ist, die Zahl der flachen Abschnitte und `vmax`. Gemerkt werden nur die Zahlen `vmin`, `vmax` und `flach`. Zeichen, die nicht zum Parcours gehören (z. B. ein Zeilenende), werden übersprungen. Mit `protokoll=True` druckt die Funktion jeden Schritt, damit wir mitverfolgen können, was passiert.

In [ ]:
BERGAB, BERGAUF, FLACH = '\\', '/', '_'      # ein Rückwärtsschrägstrich wird im Python-Code verdoppelt


def pruefe(datei, protokoll=False):
    """Liest den Parcours einmal. Rückgabe: (befahrbar, Anzahl flache Abschnitte, vmax am Ende)."""
    vmin = 0                  # kleinste mögliche Geschwindigkeit (nie negativ)
    vmax = 0                  # größte mögliche Geschwindigkeit
    flach = 0                 # Anzahl der flachen Abschnitte
    while True:
        portion = datei.read(1000000)
        if portion == '':
            break
        for zeichen in portion:
            if zeichen == BERGAB:
                vmin += 1
                vmax += 1
            elif zeichen == BERGAUF:
                vmin -= 1
                vmax -= 1
            elif zeichen == FLACH:
                vmin -= 1
                vmax += 1
                flach += 1
            else:
                continue                      # Zeilenende o. Ä.
            if vmax < 0:                      # jede mögliche Geschwindigkeit negativ: Rad bleibt stehen
                return False, flach, vmax
            if vmin < 0:                      # kleinster Wert fällt weg, nächster ist vmin + 2 (= 1)
                vmin += 2
            if protokoll:
                print(zeichen, ' vmin =', vmin, ' vmax =', vmax)
    return vmin == 0, flach, vmax


for name in ['A', 'B', 'C']:
    with open('beispieldaten/parcours' + name + '.txt') as datei:
        print('Parcours', name, ':', pruefe(datei))

Parcours A : (False, 1, -1)
Parcours B : (False, 2, 6)
Parcours C : (True, 3, 4)


Das Ergebnis `(True, 3, 4)` bei C heißt: befahrbar, 3 flache Abschnitte, `vmax = 4`. Wir schauen uns Parcours C Schritt für Schritt an (`\_\__/`):

In [ ]:
with open('beispieldaten/parcoursC.txt') as datei:
    print(pruefe(datei, protokoll=True))

\  vmin = 1  vmax = 1
_  vmin = 0  vmax = 2
\  vmin = 1  vmax = 3
_  vmin = 0  vmax = 4
_  vmin = 1  vmax = 5
/  vmin = 0  vmax = 4
(True, 3, 4)


Nach den ersten beiden Zeichen `\_` ist `vmin = 0` und `vmax = 2`: man kann bei 0 oder 2 m/s sein (der Fahrer hat auf dem flachen Stück gebremst oder beschleunigt). Am Ende ist `vmin = 0`, also ist 0 möglich: **befahrbar**. Bei Parcours A (`\_////`) wird `vmax` schon nach dem fünften Zeichen negativ, bei B fehlt am Ende das Abbremsen (`vmin` bleibt größer als 0).

Zur Probe prüfen wir die Funktion gegen **„Alles probieren“**: Für kurze Parcours (alle Zeichenfolgen bis Länge 7) probieren wir *jede* Folge von `+` und `–` auf den flachen Abschnitten aus. `fahre(text, anweisung)` simuliert eine Fahrt und sagt, ob die Geschwindigkeit nie negativ wird und am Ende 0 ist.

In [ ]:
import io
import itertools


def fahre(text, anweisung):
    """Simuliert eine Fahrt. anweisung enthält ein '+' oder '-' je flachem Abschnitt. True, wenn man sicher ankommt."""
    v = 0
    k = 0                                   # wie viele flache Abschnitte schon gefahren
    for zeichen in text:
        if zeichen == BERGAB:
            v += 1
        elif zeichen == BERGAUF:
            v -= 1
        elif zeichen == FLACH:
            if anweisung[k] == '+':
                v += 1
            else:
                v -= 1
            k += 1
        if v < 0:
            return False
    return v == 0 and k == len(anweisung)


def befahrbar_probieren(text):
    """Alles probieren: gibt es irgendeine Folge von + und -, mit der man sicher ankommt?"""
    anzahl_flach = text.count(FLACH)
    for kombination in itertools.product('+-', repeat=anzahl_flach):
        if fahre(text, kombination):
            return True
    return False


fehler = 0
anzahl = 0
for laenge in range(1, 8):
    for zeichen in itertools.product([BERGAB, BERGAUF, FLACH], repeat=laenge):
        text = ''.join(zeichen)
        anzahl += 1
        if pruefe(io.StringIO(text))[0] != befahrbar_probieren(text):
            fehler += 1
            print('Unterschied bei', text)
print(anzahl, 'Parcours verglichen, Unterschiede:', fehler)

3279 Parcours verglichen, Unterschiede: 0


Bei allen 3279 Parcours bis Länge 7 stimmt das schnelle Verfahren mit „Alles probieren“ überein.

## Schritt 2: Die Anweisungen (Teil 2)

Jetzt der zweite Durchgang. Wir kennen aus dem ersten Durchgang `flach` und `vmax`. Dann müssen `vmax // 2` flache Abschnitte `–` bekommen, und zwar die **letzten**; davor stehen `flach - vmax // 2` Mal `+`. Beim zweiten Einlesen zählen wir die flachen Abschnitte mit und geben bei jedem das passende Zeichen aus. Für lange Parcours wäre die Anweisung ellenlang; deshalb baut `anweisung` hier die Zeichenkette nur auf, und für die großen Parcours verwenden wir später nur die beiden Anzahlen.

In [ ]:
def anweisung(pfad):
    """Zwei Durchgänge über die Datei. Rückgabe: Anweisung als Text ('+', '-') oder None, wenn nicht befahrbar."""
    with open(pfad) as datei:                       # 1. Durchgang
        befahrbar, flach, vmax = pruefe(datei)
    if not befahrbar:
        return None
    bremsen = vmax // 2
    beschleunigen = flach - bremsen
    ausgabe = ''
    gesehen = 0                                     # flache Abschnitte im 2. Durchgang
    with open(pfad) as datei:                       # 2. Durchgang
        while True:
            portion = datei.read(1000000)
            if portion == '':
                break
            for zeichen in portion:
                if zeichen == FLACH:
                    gesehen += 1
                    if gesehen <= beschleunigen:
                        ausgabe += '+'
                    else:
                        ausgabe += '-'
    return ausgabe


for name in ['A', 'B', 'C']:
    print('Parcours', name, ':', anweisung('beispieldaten/parcours' + name + '.txt'))

Parcours A : None
Parcours B : None
Parcours C : +--


Bei C kommt `+--` heraus, genau wie in der Aufgabe. Dass die Anweisung wirklich stimmt, prüfen wir wieder mit der Simulation `fahre`, und zwar für alle befahrbaren Parcours bis Länge 7. Dazu schreiben wir die Parcours in eine Datei:

In [ ]:
def anweisung_text(text):
    """Wie anweisung, aber für einen Text. Zum Prüfen schreiben wir ihn in eine Datei."""
    with open('probe.txt', 'w') as datei:
        datei.write(text)
    return anweisung('probe.txt')


falsch = 0
befahrbar_anzahl = 0
for laenge in range(1, 8):
    for zeichen in itertools.product([BERGAB, BERGAUF, FLACH], repeat=laenge):
        text = ''.join(zeichen)
        a = anweisung_text(text)
        if a is not None:
            befahrbar_anzahl += 1
            if not fahre(text, a):
                falsch += 1
print(befahrbar_anzahl, 'befahrbare Parcours, falsche Anweisungen:', falsch)

256 befahrbare Parcours, falsche Anweisungen: 0


Alle Anweisungen führen sicher ins Ziel (die Datei `probe.txt` kannst du danach löschen).

**Eine Beobachtung:** Die Anweisung hängt gar nicht davon ab, wo die flachen Abschnitte liegen, sondern nur von ihrer Zahl und von `vmax`. Man müsste den Parcours also nicht ein zweites Mal lesen, es genügt, die Zahlen aus dem ersten Lauf zu behalten (dritter Merkwert: `flach`). Für den Fahrer wäre das ein Vorteil, nur der „Radfahrspaß“ fehlt: Er muss den Parcours dann selber noch einmal durchfahren.

## Die großen Parcours

Die Dateien `parcours0` bis `parcours9` haben bis zu 500 Millionen Zeichen. Sie sind nicht im Repo; mit `parcours0` können wir `pruefe` aber ausprobieren. Die anderen habe ich vorher mit demselben Verfahren berechnet (die Prüfung läuft früh ab, sobald `vmax` unter 0 fällt; bei den befahrbaren Parcours ist `vmax // 2` die Zahl der `–`):

In [ ]:
with open('beispieldaten/parcours0.txt') as datei:
    print('parcours0:', pruefe(datei))

parcours0: (False, 5, 12)


| Parcours | Länge (Zeichen) | befahrbar? | Anweisung |
|---|---:|---|---|
| parcours0 | 30 | nein | |
| parcours1 | 1 000 000 | nein | |
| parcours2 | 2 000 000 | nein | |
| parcours3 | 5 000 000 | nein | |
| parcours4 | 10 000 000 | **ja** | 1 001 577 × `+`, dann 83 × `–` |
| parcours5 | 20 000 000 | nein | |
| parcours6 | 50 000 000 | nein | |
| parcours7 | 100 000 000 | nein | |
| parcours8 | 200 000 000 | nein | |
| parcours9 | 500 000 000 | **ja** | 49 992 957 × `+`, dann 252 × `–` |

Das stimmt mit der Tabelle in den Lösungshinweisen überein. Die beiden befahrbaren Parcours haben 1 001 660 bzw. 49 993 209 flache Abschnitte (mit `vmax` = 166 bzw. 504). Das Speichern der Anweisung wäre bei `parcours9` ein Text mit 50 Millionen Zeichen, deshalb gibt man dort besser nur die beiden Zahlen aus. Die Rechenzeit wächst nur linear mit der Länge (pro Zeichen wird ein paar Mal addiert und verglichen), der Speicherbedarf bleibt unabhängig von der Länge gleich.

## Für Neugierige: Warum reicht `vmax` allein nicht?

Eine naheliegende Idee ist, nur `vmax` (und die Zahl der `+`) zu merken und am Ende `vmax // 2` Beschleunigungen in Bremsungen umzuwandeln. Das geht schief, weil man dabei nicht mehr kontrolliert, ob die Geschwindigkeit unterwegs unter 0 fällt. Der Parcours `___\\/` (Datei `eigene_testdaten/vmax_reicht_nicht.txt`) hat `vmax = 4`, ließe also zwei Bremsungen zu, aber auf drei flachen Abschnitten am Anfang kann man nicht gleich zweimal bremsen. Und tatsächlich ist er nicht befahrbar. Unser Verfahren mit `vmin` und `vmax` erkennt das:

In [ ]:
with open('eigene_testdaten/vmax_reicht_nicht.txt') as datei:
    print('mit vmin und vmax:', pruefe(datei))

# die falsche Idee: nur vmax, Befahrbarkeit = vmax gerade und >= 0
vmax = 0
for zeichen in '___\\\\/':
    if zeichen == FLACH or zeichen == BERGAB:
        vmax += 1
    else:
        vmax -= 1
print('falsche Idee (nur vmax):', 'befahrbar' if vmax % 2 == 0 and vmax >= 0 else 'nicht befahrbar', ', vmax =', vmax)
print('Alles probieren:', befahrbar_probieren('___\\\\/'))

mit vmin und vmax: (False, 3, 4)
falsche Idee (nur vmax): befahrbar , vmax = 4
Alles probieren: False


Die „falsche Idee“ hält den Parcours für befahrbar, weil `vmax = 4` gerade ist. Mit `vmin` sieht man den Fehler: Nach den drei flachen Abschnitten ist die kleinste mögliche Geschwindigkeit 1 (zweimal bremsen geht nicht, denn dabei würde die Geschwindigkeit negativ), dann folgen zwei Gefälle und nur eine Steigung, und am Ende bleiben mindestens 2 m/s übrig.

Eine Alternative, die funktioniert: nur `vmin` mitführen und zusätzlich die Zahl der bisherigen Bremsungen zählen; wird `vmin` negativ, wandelt man eine frühere Bremsung in eine Beschleunigung um. Das braucht dann auch drei Zahlen.

## Zum Schluss: Geht das mit MiniZinc?

Ja, aber nur für kurze Parcours. Wir beschreiben **nicht das Verfahren**, sondern nur die Bedingungen und lassen den Solver (Gecode) nach den `+`/`–` suchen. Für jeden Abschnitt gibt es eine Variable `v[i]` für die Geschwindigkeit danach. Am Start und am Ziel gilt `v = 0`, die Werte liegen im Bereich `0..n` (also nie negativ). Bei Steilstücken ist `v[i]` aus `v[i-1]` bestimmt, bei flachen Stücken gilt `|v[i] - v[i-1]| = 1`. Eine Lösung ist dann gleich die Fahranweisung.

```minizinc
int: n;                                   % Anzahl Abschnitte
array[1..n] of -1..1: s;                  % 1 = bergab, -1 = bergauf, 0 = flach
array[0..n] of var 0..n: v;               % Geschwindigkeit nach Abschnitt i, nie negativ

constraint v[0] = 0 /\ v[n] = 0;
constraint forall(i in 1..n)(
  if s[i] != 0 then v[i] = v[i-1] + s[i]  % Steilstück: Geschwindigkeit ist vorgegeben
  else abs(v[i] - v[i-1]) = 1 endif);     % flach: genau +1 oder -1

solve satisfy;
output [ if s[i] = 0 then (if fix(v[i] > v[i-1]) then "+" else "-" endif) else "" endif | i in 1..n ];
```

Die Datendatei erzeugt eine Python-Zelle aus dem Parcours-Text (sie ist für das Modell nur eine Zahlenliste):

In [ ]:
def als_dzn(text):
    """Parcours als MiniZinc-Daten: 1 = bergab, -1 = bergauf, 0 = flach."""
    abbildung = {BERGAB: 1, BERGAUF: -1, FLACH: 0}
    s = [abbildung[c] for c in text if c in abbildung]
    return 'n = %d;\ns = [%s];\n' % (len(s), ', '.join(str(x) for x in s))


with open('beispieldaten/parcoursC.txt') as datei:
    print(als_dzn(datei.read()))

n = 6;
s = [1, 0, 1, 0, 0, -1];



Gecode liefert für die Beispiele (der Solver gibt, wenn es eine Lösung gibt, die Anweisung aus, sonst `=====UNSATISFIABLE=====`):

Parcours A: `=====UNSATISFIABLE=====`
Parcours B: `=====UNSATISFIABLE=====`
Parcours C: `--+`
parcours0: `=====UNSATISFIABLE=====`

Das passt zu unserer Python-Lösung: A, B und `parcours0` sind nicht befahrbar, bei C findet der Solver eine gültige Anweisung (` --+ `). Das ist nicht unbedingt dieselbe wie in der Aufgabe (`+--`): Es gibt bei C mehrere richtige Anweisungen (`+--` stimmt auch, probiere es mit `fahre`). Unser Verfahren liefert eine bestimmte, der Solver irgendeine.

**Die Grenze:** Das Modell hat `n` Variablen mit je `n` möglichen Werten. Ich habe es mit allen Testparcours probiert (Gecode, Zeitlimit 60 s). A, B, C und `parcours0` sind sofort fertig. `parcours1` (1 Million Zeichen) braucht etwa eine dreiviertel Minute (Ergebnis: nicht befahrbar), bei `parcours2` bis `parcours5` (2 bis 20 Millionen Zeichen) gibt es innerhalb der 60 Sekunden gar kein Ergebnis, mit Chuffed schon bei `parcours1` nicht. Die Parcours ab 50 Millionen habe ich nicht mehr versucht; schon die Datendatei hätte hunderte Megabyte. Gegen die eine Handvoll Zahlen und den einen Durchlauf unseres Python-Verfahrens ist MiniZinc hier um Größenordnungen schlechter. Der Wert liegt woanders: Das Modell ist in zehn Zeilen aufgeschrieben, man sieht sofort, dass es das Richtige beschreibt, und es dient als **unabhängige Kontrolle** an kleinen Parcours.